# 2. StructuredTool & Tool

The **non-decorator** ways to create tools. Use these when you want to build a tool from an existing
function at runtime, wrap a function you don't own, or configure the tool explicitly.

---

## 1. Simple Definition

> **Kid version:** In tool_decorator notebooks *stuck a label* on a function with `@tool`. Sometimes you can't put a
> sticker on the function directly (maybe it's someone else's function, or you're deciding at runtime).
> So instead you **build the labeled tool by hand**, passing in the function and its label separately.

**Professional definition:**
- **`StructuredTool.from_function(...)`** builds a tool from a function with a **multi-argument** schema
  (the modern, recommended constructor).
- **`Tool(...)`** is the older, simpler class for tools that take a **single string input** (common in
  classic agents).

```python
from langchain_core.tools import StructuredTool

def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

calculator = StructuredTool.from_function(
    func=multiply,
    name="calculator",
    description="Multiply two integers together.",
)
calculator.invoke({"a": 6, "b": 7})   # 42
```

---

## 2. Why Do They Exist?

**The problem:** `@tool` is great when *you* are writing the function and can decorate it. But
sometimes:
- The function is **third-party** (you can't add a decorator to it).
- You're creating tools **dynamically** (in a loop, from config).
- You want to attach **separate sync and async** implementations.
- You want to set every field **explicitly** rather than infer it.

### Before (can't decorate a function you don't own)

```python
import some_library
# @tool  ← you can't add this to some_library.search
```

### After (`StructuredTool.from_function`)

```python
search_tool = StructuredTool.from_function(
    func=some_library.search,
    name="search",
    description="Search the external service for a query.",
)
```

They give you a **programmatic** way to build tools, decoupled from the function definition.

---

## 3. Real-Life Analogy

**Buying a labeled tool vs. labeling your own** 🔧. `@tool` is putting a sticker on a tool you made.
`StructuredTool.from_function` is going to the store, picking any tool off the shelf (even one someone
else made), and attaching your own clear label to it before handing it to a worker.

---

## 4. Where They Fit in LangChain Architecture

```
BaseTool
    │
    ├── StructuredTool     ← multi-argument tools (from_function)   ← RECOMMENDED
    └── Tool               ← single-string-input tools (older style)
```

- `@tool` is just a convenience that produces a `StructuredTool`.
- `StructuredTool` supports rich, multi-argument schemas (like `@tool`).
- `Tool` is the legacy single-input form — still seen in older agent code.

---

## 5. Internal Working

```
  StructuredTool.from_function(func=multiply, name=..., description=..., args_schema=...)
        │
        ▼
  If args_schema not given → INFER it from func's type hints (like @tool does)
        │
        ▼
  BUILD a StructuredTool with:
     name, description, args_schema, func (sync), coroutine (async, optional)
        │
        ▼
  .invoke({...}) → validates args against schema → calls func
```

---

## 6. StructuredTool — key parameters

### `func / coroutine`

**Definition:** The sync function (`func`) and/or async function (`coroutine`) the tool runs.

**Why it exists:** Lets you supply implementations explicitly, including separate async.

**When developers use it:** Always provide at least one.

```python
StructuredTool.from_function(func=sync_search, coroutine=async_search, name="search",
                             description="Search the web.")
```

---

### `name / description`

**Definition:** The tool's identifier and its purpose text (shown to the model).

**Why it exists:** Same role as with `@tool` — the model reads these to decide when/how to call.

**Real-life use case:** The label on the shelf tool.

```python
StructuredTool.from_function(func=f, name="get_stock_price",
                             description="Get the latest stock price for a ticker symbol.")
```

---

### `args_schema`

**Definition:** A Pydantic model precisely defining the arguments (names, types, descriptions,
validation).

**Why it exists:** Full control over what the model must pass; per-arg descriptions boost accuracy.

**When developers use it:** When you want validation or clear argument docs (recommended).

```python
from pydantic import BaseModel, Field

class MultiplyInput(BaseModel):
    a: int = Field(description="the first number")
    b: int = Field(description="the second number")

StructuredTool.from_function(
    func=multiply, name="multiply",
    description="Multiply two numbers.", args_schema=MultiplyInput,
)
```

---

### `return_direct`

**Definition:** If `True`, an agent returns the tool's output directly as the final answer.

```python
StructuredTool.from_function(func=f, name="lookup", description="...", return_direct=True)
```

---

## 7. The older `Tool` class (single string input)

**Definition:** `Tool` wraps a function that takes **one string** and returns a string.

**Why it exists:** Classic ReAct-style agents passed a single string to each tool. Still valid, but
`StructuredTool` is preferred for anything with multiple/typed arguments.

**Real-life use case:** A simple "give it a query string, get a string back" utility.

```python
from langchain_core.tools import Tool

def search(query: str) -> str:
    return f"results for {query}"

search_tool = Tool(
    name="search",
    description="Search the web. Input should be a search query string.",
    func=search,
)
search_tool.invoke("langchain tools")   # 'results for langchain tools'
```

> Rule of thumb: **prefer `StructuredTool`/`@tool`.** Use `Tool` only when a tool genuinely takes one
> string, or you're maintaining older code.

---

## Building tools dynamically (a real advantage)

```python
from langchain_core.tools import StructuredTool

def make_unit_converter(factor: float, name: str, desc: str):
    def convert(value: float) -> float:
        return value * factor
    return StructuredTool.from_function(func=convert, name=name, description=desc)

tools = [
    make_unit_converter(2.20462, "kg_to_lb", "Convert kilograms to pounds."),
    make_unit_converter(0.621371, "km_to_mi", "Convert kilometers to miles."),
]
# `tools` built programmatically — impossible with a static @tool decorator.
```

---

## Which creation method should I use?

| Situation | Use |
|-----------|-----|
| You're writing the function yourself | **`@tool`** — simplest |
| Function is third-party / built at runtime | **`StructuredTool.from_function`** |
| Need explicit sync + async, or full field control | `StructuredTool.from_function` |
| Legacy single-string-input tool | `Tool` |
| Need custom state, lifecycle, or overrides | subclass **`BaseTool`**  |


In [1]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama

# Initialize the language model
llm = ChatOllama(model="qwen3:8b")

d:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# step 1: define the tool with schema
class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

multiply_tool = StructuredTool.from_function(
    func = multiply,
    name = "multiply",
    description = "multiply two numbers",
    args_schema = MultiplyInput
)

C:\Users\Mr. Sachin Kapoor\AppData\Local\Temp\ipykernel_26660\3256796255.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(required=True, description="The first number to add")
C:\Users\Mr. Sachin Kapoor\AppData\Local\Temp\ipykernel_26660\3256796255.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(required=True, description="The second number to add")


In [3]:
print("=" * 40)
print("🛠️  Tool Information")
print("=" * 40)
print(f"Name        : {multiply_tool.name}")
print(f"Description : {multiply_tool.description}")
print(f"Arguments   : {multiply_tool.args}")
print("=" * 40)

🛠️  Tool Information
Name        : multiply
Description : multiply two numbers
Arguments   : {'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


In [4]:
multiply_tool.args_schema.model_json_schema()

{'properties': {'a': {'description': 'The first number to add',
   'required': True,
   'title': 'A',
   'type': 'integer'},
  'b': {'description': 'The second number to add',
   'required': True,
   'title': 'B',
   'type': 'integer'}},
 'required': ['a', 'b'],
 'title': 'MultiplyInput',
 'type': 'object'}

In [5]:
multiply_tool.invoke({"a": 5, "b": 3})

15

In [6]:
#step2: bind the tool to the language model
llm_with_tools = llm.bind_tools([multiply_tool])

In [7]:
# step3: tool calling
msg = llm_with_tools.invoke("What's 12 times 7?")
msg

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-08-08T14:40:24.9012262Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11868573700, 'load_duration': 107753600, 'prompt_eval_count': 158, 'prompt_eval_duration': 534006999, 'eval_count': 152, 'eval_duration': 11191529000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--019fe1d1-38e6-75b1-84c5-658f728c79c1-0', tool_calls=[{'name': 'multiply', 'args': {'a': 12, 'b': 7}, 'id': '4aedd70a-aeae-4b90-b89e-fcfcb663c5c8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 158, 'output_tokens': 152, 'total_tokens': 310})

In [8]:
msg.tool_calls

[{'name': 'multiply',
  'args': {'a': 12, 'b': 7},
  'id': '4aedd70a-aeae-4b90-b89e-fcfcb663c5c8',
  'type': 'tool_call'}]

In [9]:
# step 4: tool execution
result = multiply_tool.invoke(llm_with_tools.invoke("can you multiply 3 with 10").tool_calls[0]['args'])
result

30

In [10]:
from langchain_core.tools import StructuredTool
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

# STEP 1: Define input schemas
class AddInput(BaseModel):
    a: float = Field(description="First number")
    b: float = Field(description="Second number")


class SubtractInput(BaseModel):
    a: float = Field(description="First number")
    b: float = Field(description="Second number")


class MultiplyInput(BaseModel):
    a: float = Field(description="First number")
    b: float = Field(description="Second number")


class DivideInput(BaseModel):
    a: float = Field(description="Numerator")
    b: float = Field(description="Denominator")

In [11]:
# STEP 2: Define normal Python functions
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b


def subtract(a: float, b: float) -> float:
    """Subtract b from a."""
    return a - b


def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b


def divide(a: float, b: float) -> float:
    """Divide a by b."""
    if b == 0:
        return "Cannot divide by zero"
    return a / b

In [12]:
# STEP 3: Create StructuredTools
add_tool = StructuredTool.from_function(
    func=add,
    name="add",
    description="Add two numbers.",
    args_schema=AddInput
)

subtract_tool = StructuredTool.from_function(
    func=subtract,
    name="subtract",
    description="Subtract b from a.",
    args_schema=SubtractInput
)

multiply_tool = StructuredTool.from_function(
    func=multiply,
    name="multiply",
    description="Multiply two numbers.",
    args_schema=MultiplyInput
)

divide_tool = StructuredTool.from_function(
    func=divide,
    name="divide",
    description="Divide a by b.",
    args_schema=DivideInput
)

In [13]:
tools = [add_tool, subtract_tool, multiply_tool, divide_tool]

llm = ChatOllama(model="qwen3:8b")

In [14]:
# STEP 4: Bind tools to LLM
llm_with_tools = llm.bind_tools(tools)

In [15]:
for tool in tools:
    print("=" * 50)
    print(f"Tool       : {tool.name}")
    print(f"Description: {tool.description}")
    print(f"Arguments  : {tool.args}")
    print("Schema JSON:")
    print(tool.args_schema.model_json_schema())
    print("=" * 50)

Tool       : add
Description: Add two numbers.
Arguments  : {'a': {'description': 'First number', 'title': 'A', 'type': 'number'}, 'b': {'description': 'Second number', 'title': 'B', 'type': 'number'}}
Schema JSON:
{'properties': {'a': {'description': 'First number', 'title': 'A', 'type': 'number'}, 'b': {'description': 'Second number', 'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'AddInput', 'type': 'object'}
Tool       : subtract
Description: Subtract b from a.
Arguments  : {'a': {'description': 'First number', 'title': 'A', 'type': 'number'}, 'b': {'description': 'Second number', 'title': 'B', 'type': 'number'}}
Schema JSON:
{'properties': {'a': {'description': 'First number', 'title': 'A', 'type': 'number'}, 'b': {'description': 'Second number', 'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'SubtractInput', 'type': 'object'}
Tool       : multiply
Description: Multiply two numbers.
Arguments  : {'a': {'description': 'First number', 'title':

In [16]:
# STEP 5: Invoke the LLM with tools
response = llm_with_tools.invoke("What is 25 multiplied by 12?")
response

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-08-08T14:40:46.9003807Z', 'done': True, 'done_reason': 'stop', 'total_duration': 13424664300, 'load_duration': 105187700, 'prompt_eval_count': 313, 'prompt_eval_duration': 83305000, 'eval_count': 169, 'eval_duration': 13214244000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--019fe1d1-88c2-7902-b272-3387b9276423-0', tool_calls=[{'name': 'multiply', 'args': {'a': 25, 'b': 12}, 'id': 'cccc572c-0b01-40ec-b285-a645312239d9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 313, 'output_tokens': 169, 'total_tokens': 482})

In [17]:
response.tool_calls

[{'name': 'multiply',
  'args': {'a': 25, 'b': 12},
  'id': 'cccc572c-0b01-40ec-b285-a645312239d9',
  'type': 'tool_call'}]

In [18]:
print("LLM requested:")
print(response.tool_calls)

# Execute the requested tool
tool_call = response.tool_calls[0]

tool_name = tool_call["name"]
tool_args = tool_call["args"]

for tool in tools:
    if tool.name == tool_name:
        tool_result = tool.invoke(tool_args)
        break

print("\nTool result:")
print(tool_result)

LLM requested:
[{'name': 'multiply', 'args': {'a': 25, 'b': 12}, 'id': 'cccc572c-0b01-40ec-b285-a645312239d9', 'type': 'tool_call'}]

Tool result:
300.0
